# ZeroMRG on COV-CTR: guided Kaggle reproduction

This notebook is for **COV-CTR English reports only**. COV-CTR is different from the paper's COV-CT Chinese-report dataset. It implements the stated flow in Jiang & Chang, *Auto-encoding clinical language for zero-shot medical report generation* (2026): text-only auto-encoding, 500 training-report prompts, image-to-prompt attention, and optional 10%/100% paired training.

**Exact paper reproduction is impossible from the disclosed information.** The paper does not specify the decoder internals, tokenizer, seed/split membership, full preprocessing, decoding search, or exact metric-toolkit version. The supplied COV-CTR archive has 714 unambiguous image/report pairs, while the paper reports 728 images. We mark necessary choices as **ASSUMPTION** and save actual results separately from Table 1's published numbers. No values are inserted as experimental output.

Run the cells in order. Section 9 produces the **0% paired** score. Sections 10 and 11 train **10%** and **100%** paired models independently from the same text-only decoder. They can take much longer and need more GPU memory. A score measures text overlap; read the generated reports before drawing conclusions about medical quality.

## 1. Set up Kaggle

Import this notebook into Kaggle, attach a Dataset containing `COV-CTR/reports_ZH_EN.csv` and its images (or the original ZIP), and select a GPU. Enable Internet for the first package/model download. The cell installs the official OpenAI CLIP package, the M-CLIP text encoder, and the COCO caption metrics. If package imports fail after installation, restart the Kaggle session once, then run from the top.

In [ ]:
%pip -q install "multilingual-clip>=1.0.10,<2" "transformers>=4.35,<5" sentencepiece "pycocoevalcap>=1.2,<2" "git+https://github.com/openai/CLIP.git"

In [ ]:
import json, math, random, re, shutil, unicodedata, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42                 # ASSUMPTION: paper does not state a seed.
BATCH_SIZE = 6            # PAPER: mini-batch size.
MAX_EPOCHS = 200          # ASSUMPTION: paper gives no maximum epoch count.
PATIENCE = 50             # PAPER: stop after 50 epochs without validation BLEU gain.
PROMPT_COUNT = 500        # PAPER: COV-CTR prompt count.
MAX_TOKENS = 160          # ASSUMPTION: enough for every report in supplied archive.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Choose a GPU accelerator in Kaggle Notebook settings."

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/zeromrg_cov_ctr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))
print("Outputs:", OUTPUT_DIR)

## 2. Match images to English reports and audit exclusions

The English target is `reports_En`, joined to an image by the exact `image_id` filename. We exclude all duplicate IDs with conflicting reports instead of choosing a report arbitrarily, and exclude images with no report. **ASSUMPTION:** that conservative rule produces 714 usable pairs in the supplied archive. The paper does not disclose how its 728-image population was constructed.

In [ ]:
def find_csv():
    direct = list(INPUT_DIR.rglob("reports_ZH_EN.csv"))
    if len(direct) == 1:
        return direct[0]
    if len(direct) > 1:
        raise RuntimeError(f"Found multiple report CSV files: {direct}")
    archives = []
    for path in INPUT_DIR.rglob("*.zip"):
        with zipfile.ZipFile(path) as z:
            if any(Path(member).name == "reports_ZH_EN.csv" for member in z.namelist()):
                archives.append(path)
    if len(archives) != 1:
        raise FileNotFoundError("Attach one COV-CTR Dataset with reports_ZH_EN.csv or its ZIP.")
    location = OUTPUT_DIR / "extracted_input"
    location.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(location)  # Only extract the COV-CTR Dataset that you attached.
    found = list(location.rglob("reports_ZH_EN.csv"))
    if len(found) != 1:
        raise RuntimeError("ZIP must contain exactly one reports_ZH_EN.csv")
    return found[0]

CSV_PATH = find_csv()
COV_ROOT = CSV_PATH.parent
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
assert {"image_id", "reports_En"}.issubset(raw.columns), list(raw.columns)

def normalize_report(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(value))).strip()

raw["image_id"] = raw["image_id"].astype(str).str.strip()
raw["report"] = raw["reports_En"].map(normalize_report)
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = [p for p in COV_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
by_name = defaultdict(list)
for path in images:
    by_name[path.name.lower()].append(path)

rows = []
excluded = Counter()
for image_id, group in raw.groupby("image_id", sort=True):
    if len(group) != 1:
        excluded["duplicate_image_id"] += 1
        continue
    report = group.iloc[0].report
    if not report:
        excluded["empty_report"] += 1
        continue
    paths = by_name.get(image_id.lower(), [])
    if len(paths) != 1:
        excluded["image_missing_or_ambiguous"] += 1
        continue
    try:
        with Image.open(paths[0]) as image:
            image.verify()
    except Exception:
        excluded["unreadable_image"] += 1
        continue
    rows.append({"image_id": image_id, "image_path": str(paths[0]), "report": report})

manifest = pd.DataFrame(rows)
assert len(manifest) >= 630, "Too few pairs to create 500 training prompts."
print("CSV rows:", len(raw), "physical images:", len(images))
print("Valid pairs:", len(manifest), "excluded IDs:", dict(excluded))
if len(manifest) != 714:
    print("NOTICE: attached archive differs from the audited 714-pair archive.")
display(manifest.head(3))

## 3. Create the 8:1:1 split and 500 training prompts

**PAPER:** random 8:1:1 train/validation/test split and 500 prompts sampled from the training reports. **ASSUMPTION:** seed 42 and unique-image-ID splitting. The split and selected prompts are written to output files so a run is auditable. No validation/test report is used as a training prompt.

In [ ]:
permutation = np.random.default_rng(SEED).permutation(len(manifest))
n_train = round(0.8 * len(manifest))
n_val = round(0.1 * len(manifest))
train_df = manifest.iloc[permutation[:n_train]].reset_index(drop=True)
val_df = manifest.iloc[permutation[n_train:n_train+n_val]].reset_index(drop=True)
test_df = manifest.iloc[permutation[n_train+n_val:]].reset_index(drop=True)
assert len(train_df) >= PROMPT_COUNT
train_ids, val_ids, test_ids = map(lambda d: set(d.image_id), (train_df, val_df, test_df))
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)
prompt_df = train_df.sample(n=PROMPT_COUNT, random_state=SEED).reset_index(drop=True)
assert set(prompt_df.image_id) <= train_ids
print("train / validation / test:", len(train_df), len(val_df), len(test_df))
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df), ("prompts", prompt_df)):
    frame[["image_id", "report"]].to_csv(OUTPUT_DIR / f"{name}_manifest.csv", index=False)

## 4. Build the decoder vocabulary from training reports only

M-CLIP has its own pretrained tokenizer. The paper does **not** specify the report decoder tokenizer. **ASSUMPTION:** preserve word case, separate punctuation, and use `<bos>`, `<eos>`, `<pad>`, and `<unk>`. A long report causes an error rather than silently losing its end token.

In [ ]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>"]
def tokenize(text):
    return TOKEN_RE.findall(text)

counts = Counter(t for report in train_df.report for t in tokenize(report))
vocab = SPECIAL + sorted(counts)
stoi = {token: idx for idx, token in enumerate(vocab)}
PAD, BOS, EOS, UNK = range(4)

def encode_report(text):
    ids = [BOS] + [stoi.get(token, UNK) for token in tokenize(text)] + [EOS]
    if len(ids) > MAX_TOKENS:
        raise ValueError(f"A report has {len(ids)} tokens. Increase MAX_TOKENS.")
    return ids

def decode_report(ids):
    tokens = []
    for item in ids:
        item = int(item)
        if item == EOS:
            break
        if item not in (BOS, PAD):
            tokens.append(vocab[item])
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    return re.sub(r"([(\[])\s+", r"\1", text).strip()

lengths = [len(encode_report(text)) for text in manifest.report]
print("Vocab size:", len(vocab), "longest report:", max(lengths), "token limit:", MAX_TOKENS)
(OUTPUT_DIR / "decoder_vocabulary.json").write_text(json.dumps(vocab, ensure_ascii=False), encoding="utf-8")

## 5. Load paper's M-CLIP and corresponding OpenAI CLIP image model

**PAPER:** use `M-CLIP/XLM-Roberta-Large-Vit-L-14`. The official checkpoint identifies OpenAI CLIP `ViT-L/14` as its companion image encoder. M-CLIP text weights remain frozen; OpenAI CLIP's official image preprocessing is used. The test below verifies the image and text features share the expected 768-dimensional space.

In [ ]:
from multilingual_clip import pt_multilingual_clip
from transformers import AutoTokenizer
import clip

MCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"
text_model = pt_multilingual_clip.MultilingualCLIP.from_pretrained(MCLIP_NAME).to(DEVICE).eval()
text_tokenizer = AutoTokenizer.from_pretrained(MCLIP_NAME)
for parameter in text_model.parameters():
    parameter.requires_grad_(False)

full_clip, clip_preprocess = clip.load("ViT-L/14", device=DEVICE, jit=False)
vision = full_clip.visual.eval()
for parameter in vision.parameters():
    parameter.requires_grad_(False)
del full_clip
torch.cuda.empty_cache()

@torch.no_grad()
def encode_text_raw(texts):
    return text_model.forward(list(texts), text_tokenizer).float()

def encode_image_raw(images):
    dtype = next(vision.parameters()).dtype
    return vision(images.to(DEVICE, dtype=dtype)).float()

text_probe = encode_text_raw(["No acute abnormality."])
with torch.no_grad():
    image_probe = encode_image_raw(torch.zeros(1, 3, 224, 224))
assert text_probe.shape == image_probe.shape == (1, 768), (text_probe.shape, image_probe.shape)
D_MODEL = 768
print("M-CLIP text:", text_probe.shape, "OpenAI CLIP image:", image_probe.shape)

# Keep the vision tower on CPU while training only the decoder.
vision.to("cpu")
torch.cuda.empty_cache()

## 6. Dataset batches and transformer decoder

**PAPER:** a transformer-based decoder trained by next-token cross-entropy. **ASSUMPTION:** six decoder layers, eight attention heads, width 768, feed-forward width 3072, dropout 0.1, learned positions, and greedy generation. The single report/image feature is supplied through cross-attention. A causal mask hides future output tokens.

In [ ]:
class CovDataset(Dataset):
    def __init__(self, frame, with_images=False):
        self.frame = frame.reset_index(drop=True)
        self.with_images = with_images
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        example = {"image_id": row.image_id, "report": row.report,
                   "ids": torch.tensor(encode_report(row.report), dtype=torch.long)}
        if self.with_images:
            with Image.open(row.image_path) as image:
                example["image"] = clip_preprocess(image.convert("RGB"))
        return example

def collate(examples):
    width = max(len(x["ids"]) for x in examples)
    ids = torch.full((len(examples), width), PAD, dtype=torch.long)
    for index, example in enumerate(examples):
        ids[index, :len(example["ids"])] = example["ids"]
    result = {"image_id": [x["image_id"] for x in examples],
              "report": [x["report"] for x in examples], "ids": ids}
    if "image" in examples[0]:
        result["images"] = torch.stack([x["image"] for x in examples])
    return result

def make_loader(frame, with_images=False, shuffle=False):
    return DataLoader(CovDataset(frame, with_images), batch_size=BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate, num_workers=0)

class ReportDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(len(vocab), D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_TOKENS, D_MODEL)
        layer = nn.TransformerDecoderLayer(D_MODEL, 8, 3072, 0.1, "gelu", batch_first=True)
        self.transformer = nn.TransformerDecoder(layer, num_layers=6)
        self.output = nn.Linear(D_MODEL, len(vocab))
    def forward(self, condition, input_ids):
        _, length = input_ids.shape
        positions = torch.arange(length, device=input_ids.device).unsqueeze(0)
        target = self.tokens(input_ids) + self.positions(positions)
        causal_mask = torch.triu(torch.ones(length, length, device=input_ids.device, dtype=torch.bool), 1)
        hidden = self.transformer(target, condition.unsqueeze(1), tgt_mask=causal_mask,
                                  tgt_key_padding_mask=input_ids.eq(PAD))
        return self.output(hidden)
    @torch.no_grad()
    def generate(self, condition):
        self.eval()
        ids = torch.full((condition.size(0), 1), BOS, device=condition.device, dtype=torch.long)
        finished = torch.zeros(condition.size(0), device=condition.device, dtype=torch.bool)
        for _ in range(MAX_TOKENS - 1):
            next_id = self(condition, ids)[:, -1].argmax(dim=-1)
            next_id = torch.where(finished, torch.full_like(next_id, PAD), next_id)
            ids = torch.cat((ids, next_id[:, None]), dim=1)
            finished |= next_id.eq(EOS)
            if bool(finished.all()):
                break
        return ids

decoder = ReportDecoder().to(DEVICE)
example_batch = next(iter(make_loader(train_df)))
print("Token batch:", tuple(example_batch["ids"].shape),
      "decoder parameters:", sum(p.numel() for p in decoder.parameters()))

## 7. Train the decoder with reports alone (0% paired)

Each training report is encoded by frozen M-CLIP and L2-normalized, as in Eq. (3). We cache those fixed vectors once to avoid repeating the same expensive encoder pass every epoch. **PAPER:** SGD, learning rate 0.001, momentum 0.99, batch size 6, cross-entropy, dropout, and BLEU early stopping with patience 50. **ASSUMPTION:** validation BLEU means BLEU-4; gradient clipping at 5; maximum 200 epochs. Only training reports update the decoder. Validation reports select a checkpoint; the test set is untouched here.

In [ ]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

assert shutil.which("java"), "The COCO PTB tokenizer needs Java in the Kaggle environment."
ptb = PTBTokenizer()

def score_reports(references, predictions, bleu_only=False):
    assert len(references) == len(predictions) and references
    refs = {i: [{"caption": r}] for i, r in enumerate(references)}
    preds = {i: [{"caption": p}] for i, p in enumerate(predictions)}
    refs, preds = ptb.tokenize(refs), ptb.tokenize(preds)
    bleu, _ = Bleu(4).compute_score(refs, preds)
    scores = {f"BLEU-{i+1}": 100 * float(value) for i, value in enumerate(bleu)}
    if not bleu_only:
        scores["CIDEr"] = 100 * float(Cider().compute_score(refs, preds)[0])
        scores["ROUGE-L"] = 100 * float(Rouge().compute_score(refs, preds)[0])
    return scores

@torch.no_grad()
def cached_text_features(frame):
    result = {}
    # Encoder batch 2 is only a memory-saving inference choice; training batch remains 6.
    for start in range(0, len(frame), 2):
        part = frame.iloc[start:start+2]
        vectors = F.normalize(encode_text_raw(part.report.tolist()), dim=-1).cpu()
        result.update(zip(part.image_id, vectors))
    return result

train_text = cached_text_features(train_df)
val_text = cached_text_features(val_df)

# Prompt vectors use the same frozen text model. The paper does not state whether
# these are normalized before Eq. (6); raw prompt features are an ASSUMPTION.
@torch.no_grad()
def build_prompt_bank(frame):
    parts = [encode_text_raw(frame.report.iloc[i:i+2].tolist()).cpu()
             for i in range(0, len(frame), 2)]
    return torch.cat(parts).to(DEVICE)

prompts = build_prompt_bank(prompt_df)
assert prompts.shape == (PROMPT_COUNT, D_MODEL)
text_model.to("cpu")
torch.cuda.empty_cache()

def loss_sum(condition, ids):
    logits = decoder(condition, ids[:, :-1])
    target = ids[:, 1:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1),
                           ignore_index=PAD, reduction="sum")

@torch.no_grad()
def validate_text_bleu4():
    decoder.eval()
    references, predictions = [], []
    for batch in make_loader(val_df):
        features = torch.stack([val_text[key] for key in batch["image_id"]]).to(DEVICE)
        output = decoder.generate(features).cpu()
        references.extend(batch["report"])
        predictions.extend(decode_report(row) for row in output)
    return score_reports(references, predictions, bleu_only=True)["BLEU-4"]

optimizer = torch.optim.SGD(decoder.parameters(), lr=1e-3, momentum=0.99)
text_checkpoint = OUTPUT_DIR / "text_decoder_best.pt"
best_text_bleu, bad_epochs = -1.0, 0

# Check a real forward and backward pass before the long run.
condition = torch.stack([train_text[key] for key in example_batch["image_id"]]).to(DEVICE)
ids = example_batch["ids"].to(DEVICE)
probe_loss = loss_sum(condition, ids) / ids[:, 1:].ne(PAD).sum()
assert bool(torch.isfinite(probe_loss)), "Initial loss is not finite."
probe_loss.backward()
optimizer.zero_grad(set_to_none=True)
print("Finite initial loss:", float(probe_loss.detach()))

for epoch in range(1, MAX_EPOCHS + 1):
    decoder.train()
    losses = []
    for batch in make_loader(train_df, shuffle=True):
        features = torch.stack([train_text[key] for key in batch["image_id"]]).to(DEVICE)
        ids = batch["ids"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_sum(features, ids) / ids[:, 1:].ne(PAD).sum()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 5.0)
        optimizer.step()
        losses.append(float(loss.detach()))
    val_bleu4 = validate_text_bleu4()
    print(f"text epoch {epoch:03d}: CE={np.mean(losses):.4f}, val BLEU-4={val_bleu4:.2f}")
    if val_bleu4 > best_text_bleu:
        best_text_bleu, bad_epochs = val_bleu4, 0
        torch.save(decoder.state_dict(), text_checkpoint)
    else:
        bad_epochs += 1
    if bad_epochs >= PATIENCE:
        print("Early stopping after 50 non-improving epochs.")
        break

decoder.load_state_dict(torch.load(text_checkpoint, map_location=DEVICE, weights_only=True))
decoder.eval()
print("Best text-validation BLEU-4:", best_text_bleu)

## 8. Combine image features with 500 fixed report prompts

**PAPER Eqs. (5)–(6):** image features query the prompt bank using scaled dot-product attention; the attended prompt is added to the image feature; then the sum is L2-normalized. We do **not** normalize image/prompt vectors before that attention step, because Eq. (5) places normalization after addition. Prompt normalization before Eq. (6) is unspecified, so raw M-CLIP prompt vectors are an **ASSUMPTION**. The decoder and both encoders are frozen for the 0% experiment.

In [ ]:
vision.to(DEVICE).eval()

def prompt_enrich(image_vector):
    weights = torch.softmax(image_vector @ prompts.T / math.sqrt(D_MODEL), dim=-1)
    retrieved = weights @ prompts
    return F.normalize(image_vector + retrieved, p=2, dim=-1)

@torch.no_grad()
def image_condition(images):
    return prompt_enrich(encode_image_raw(images))

@torch.no_grad()
def predict(frame):
    decoder.eval(); vision.eval()
    records = []
    for batch in make_loader(frame, with_images=True):
        generated = decoder.generate(image_condition(batch["images"])).cpu()
        records.extend({"image_id": key, "reference": ref, "generated": decode_report(ids)}
                       for key, ref, ids in zip(batch["image_id"], batch["report"], generated))
    return pd.DataFrame(records)

test_batch = next(iter(make_loader(test_df, with_images=True)))
test_condition = image_condition(test_batch["images"])
assert test_condition.shape == (BATCH_SIZE, D_MODEL)
assert torch.allclose(test_condition.norm(dim=-1), torch.ones(BATCH_SIZE, device=DEVICE), atol=1e-3)
print("prompt bank:", tuple(prompts.shape), "image condition:", tuple(test_condition.shape))

## 9. Evaluate held-out test images: **0% paired**

Metrics are computed with the COCO caption toolkit's PTB tokenizer, BLEU, CIDEr, and ROUGE-L, then shown on the paper's 0–100 scale. This follows the toolkit family specified in the paper; its exact version/tokenization details were not published. `predictions_0pct.csv` contains the reports actually generated by your run.

In [ ]:
predictions_0 = predict(test_df)
scores_0 = score_reports(predictions_0.reference.tolist(), predictions_0.generated.tolist())
predictions_0.to_csv(OUTPUT_DIR / "predictions_0pct.csv", index=False)
(OUTPUT_DIR / "metrics_0pct.json").write_text(json.dumps(scores_0, indent=2), encoding="utf-8")
print("0% paired TEST scores:", scores_0)
display(predictions_0.head(5))

## 10. Prepare the 10% and 100% paired experiments

**PAPER:** fine-tune the image encoder and decoder with image/report pairs and cross-entropy. The paper does not say precisely which visual weights are unfrozen or how the supervised stages start. **ASSUMPTION:** train the full OpenAI CLIP visual tower and decoder; start **each** paired experiment from the same 0% text-decoder checkpoint and original CLIP vision weights. Prompts remain fixed. To fit typical Kaggle GPU memory, each update processes six examples as microbatches of one with accumulated gradients. This preserves the six-example loss for the update but is a documented engineering adaptation. It can still run out of memory or exceed session limits.

In [ ]:
# Save initial states on CPU once, before either paired stage changes them.
text_start = {key: value.detach().cpu().clone() for key, value in decoder.state_dict().items()}
vision_start = {key: value.detach().cpu().clone() for key, value in vision.state_dict().items()}

@torch.no_grad()
def validate_image_bleu4():
    frame = predict(val_df)
    return score_reports(frame.reference.tolist(), frame.generated.tolist(), bleu_only=True)["BLEU-4"]

def train_paired(fraction, name):
    decoder.load_state_dict(text_start)
    vision.load_state_dict(vision_start)
    vision.float()  # Full-precision trainable weights; AMP handles activations.
    for parameter in vision.parameters():
        parameter.requires_grad_(True)
    frame = train_df if fraction == 1.0 else train_df.sample(
        n=round(fraction * len(train_df)), random_state=SEED).reset_index(drop=True)
    print(name, "training pairs:", len(frame))
    optimizer = torch.optim.SGD(list(vision.parameters()) + list(decoder.parameters()),
                                lr=1e-3, momentum=0.99)
    scaler = torch.cuda.amp.GradScaler()
    best_bleu, bad_epochs = -1.0, 0
    checkpoint = OUTPUT_DIR / f"paired_{name}_best.pt"
    for epoch in range(1, MAX_EPOCHS + 1):
        decoder.train(); vision.train()
        epoch_losses = []
        for batch in make_loader(frame, with_images=True, shuffle=True):
            optimizer.zero_grad(set_to_none=True)
            all_ids = batch["ids"].to(DEVICE)
            total_tokens = all_ids[:, 1:].ne(PAD).sum()
            batch_loss = 0.0
            for index in range(len(all_ids)):
                images = batch["images"][index:index+1].to(DEVICE)
                ids = all_ids[index:index+1]
                with torch.cuda.amp.autocast():
                    image_vector = encode_image_raw(images)
                    condition = prompt_enrich(image_vector)
                    loss = loss_sum(condition, ids) / total_tokens
                scaler.scale(loss).backward()
                batch_loss += float(loss.detach())
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(vision.parameters()) + list(decoder.parameters()), 5.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(batch_loss)
        val_bleu4 = validate_image_bleu4()
        print(f"{name} epoch {epoch:03d}: CE={np.mean(epoch_losses):.4f}, val BLEU-4={val_bleu4:.2f}")
        if val_bleu4 > best_bleu:
            best_bleu, bad_epochs = val_bleu4, 0
            torch.save({"decoder": decoder.state_dict(), "vision": vision.state_dict()}, checkpoint)
        else:
            bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping after 50 non-improving epochs.")
            break
    saved = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
    decoder.load_state_dict(saved["decoder"])
    vision.load_state_dict(saved["vision"])
    predictions = predict(test_df)
    scores = score_reports(predictions.reference.tolist(), predictions.generated.tolist())
    predictions.to_csv(OUTPUT_DIR / f"predictions_{name}.csv", index=False)
    (OUTPUT_DIR / f"metrics_{name}.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
    print(name, "TEST scores:", scores)
    return scores

## 11. Train and evaluate **10% paired**

Only about 57 of the 571 training images are used here. Validation/test membership and the 500 prompts stay the same as in 0%.

In [ ]:
scores_10 = train_paired(0.10, '10pct')

## 12. Train and evaluate **100% paired**

This function resets both models to their initial states, then uses all 571 training pairs. Run it after 10%, or run it directly if you only need 0% and 100%.

In [ ]:
scores_100 = train_paired(1.0, '100pct')

## 13. Compare your actual results with Table 1

The printed paper numbers below are **reference values**, not measured output. Exact matching is not expected because paper details and the 728-image population are unavailable. Look at the prediction CSVs as well as the numbers. The files under `/kaggle/working/zeromrg_cov_ctr/` are your real outputs; save a Kaggle notebook version with outputs or download them before the session ends.

In [ ]:
paper = pd.DataFrame([
    {"paired": "0%", "BLEU-1": 65.9, "BLEU-2": 58.5, "BLEU-3": 54.8, "BLEU-4": 50.7, "CIDEr": 63.5, "ROUGE-L": 67.4},
    {"paired": "10%", "BLEU-1": 68.8, "BLEU-2": 62.1, "BLEU-3": 59.5, "BLEU-4": 53.6, "CIDEr": 66.5, "ROUGE-L": 72.3},
    {"paired": "100%", "BLEU-1": 75.1, "BLEU-2": 68.6, "BLEU-3": 64.4, "BLEU-4": 61.2, "CIDEr": 72.3, "ROUGE-L": 77.5},
])
actual = pd.DataFrame([{"paired": "0%", **scores_0}, {"paired": "10%", **scores_10},
                       {"paired": "100%", **scores_100}])
print("PUBLISHED PAPER TABLE (reference only)")
display(paper)
print("THIS RUN (computed from held-out predictions)")
display(actual.round(2))
paper.merge(actual, on="paired", suffixes=("_paper", "_actual")).to_csv(
    OUTPUT_DIR / "comparison_with_paper.csv", index=False)